[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/SFM/blob/main/notebooks/EN/chapter6_seminar_notebook.ipynb)

# Statistics of Financial Markets — Seminar 6: Model selection and risk management

*Seminar notebook. Daniel Traian Pele, Bucharest University of Economic Studies.*

- The slides "Prerequisites for Today" give every definition used here.
- Part A: computations on paper, checked in code. Part B: real data with an interpretation question. Part C: an open question and an AI answer to audit.
- **[Solved]**: complete code and output, a model to follow. **[Proposed]**: write your own code in the empty cell.

> The solutions are discussed in the seminar.

> **Working in Google Colab? Save your own copy first.**
> The Colab link opens a temporary copy: when you close the tab or the session times out, your changes are lost.
> Use **File → Save a copy in Drive** (it goes to the *Colab Notebooks* folder of your ASE Google Drive) and work in that copy.
> For the team project, **File → Save a copy in GitHub** puts the notebook directly in your team's repository.
> Files you create while the notebook runs (CSV, charts) are also temporary: set `SAVE_TO_DRIVE = True` in the next cell to keep them.

In [ ]:
# Optional (Colab only): keep the files you create in Google Drive
SAVE_TO_DRIVE = False   # set to True, run this cell and allow access to your Drive

import os
OUT_DIR = '.'
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/SFM/Chapter_6'
    os.makedirs(OUT_DIR, exist_ok=True)
print('Save your outputs to:', os.path.abspath(OUT_DIR))


In [ ]:
import os
import re
import json
import urllib.request
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgb
from scipy import optimize
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

## Chart style

- Transparent background, legend below the plot, course palette.

In [ ]:
# Palette (RGB values of latex/preamble.tex)
MainBlue = '#1A3A6E'
IDAred = '#CD0000'
Forest = '#2E7D32'
Amber = '#B5853F'
Orange = '#E67E22'
Purple = '#8E44AD'
Teal = '#17A2B8'
Crimson = '#DC3545'
DarkText = '#1F2A44'          # text, axes and ticks (dark navy, not grey)

PALETTE = [MainBlue, IDAred, Forest, Amber, Purple, Orange, Teal, Crimson]
# fixed colours for the series used in several chapters
COL = {'sp500': MainBlue, 'bet': IDAred, 'bettr': Orange, 'dax': Teal, 'btc': Amber, 'eth': Crimson,
       'eurron': Forest, 'gold': Purple, 'vix': Crimson, 'stoxx50': Forest, 'ndx': Teal}


def apply():
    """Set the course style for matplotlib."""
    rc = plt.rcParams
    rc['figure.facecolor'] = 'none'
    rc['axes.facecolor'] = 'none'
    rc['savefig.facecolor'] = 'none'
    rc['savefig.transparent'] = True
    rc['axes.grid'] = False
    rc['font.family'] = 'sans-serif'
    rc['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
    # font sizes for slides (charts are 9-11 inches wide and shown at 8-14 cm)
    rc['font.size'] = 12
    rc['axes.labelsize'] = 13
    rc['axes.titlesize'] = 13
    rc['xtick.labelsize'] = 11.5
    rc['ytick.labelsize'] = 11.5
    rc['legend.fontsize'] = 11
    rc['axes.spines.top'] = False
    rc['axes.spines.right'] = False
    rc['axes.linewidth'] = 0.6
    rc['lines.linewidth'] = 1.4
    rc['legend.facecolor'] = 'none'
    rc['legend.framealpha'] = 0
    rc['legend.frameon'] = False
    for k in ('text.color', 'axes.labelcolor', 'axes.edgecolor', 'xtick.color', 'ytick.color', 'axes.titlecolor'):
        rc[k] = DarkText
    rc['axes.prop_cycle'] = mpl.cycler(color=PALETTE)


def legend_outside_bottom(ax, ncol=2, y=-0.22, **kw):
    """Place the legend outside the plot, bottom centre (for a figure with several axes, pass the last one
    or use fig.legend with the same arguments)."""
    return ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False, **kw)


def fig_legend_bottom(fig, handles=None, labels=None, ncol=3, y=-0.02):
    """One legend for a whole figure (several panels), below the panels."""
    if handles is None:
        handles, labels = [], []
        for ax in fig.axes:
            for h, l in zip(*ax.get_legend_handles_labels()):
                if l not in labels and not l.startswith('_'):
                    handles.append(h)
                    labels.append(l)
    return fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


def _is_grey(c, tol=0.06):
    try:
        r, g, b = to_rgb(c)
    except ValueError:
        return False
    return max(r, g, b) - min(r, g, b) < tol and 0.25 < (r + g + b) / 3 < 0.95


def check_no_grey(fig):
    """House rule: no grey series and no grey text. Raises ValueError listing the offending elements."""
    bad = []
    for ax in fig.axes:
        for ln in ax.get_lines():
            if not ln.get_label().startswith('_') and _is_grey(ln.get_color()):
                bad.append(f'line {ln.get_label()!r}')
        for coll in ax.collections:
            fc = coll.get_facecolor()
            if len(fc) and coll.get_label() and not coll.get_label().startswith('_') and _is_grey(fc[0][:3]):
                bad.append(f'series {coll.get_label()!r}')
        for t in ax.texts + [ax.title, ax.xaxis.label, ax.yaxis.label]:
            if t.get_text() and _is_grey(t.get_color()):
                bad.append(f'text {t.get_text()[:30]!r}')
    if bad:
        raise ValueError('grey elements: ' + ', '.join(bad))


def save_fig(name, out_dir=None, show=True):
    """Save the figure as transparent PDF and PNG (OUT_DIR if defined, else the current folder), then show it."""
    d = out_dir or globals().get('OUT_DIR', '.')
    plt.savefig(os.path.join(d, f'{name}.pdf'), bbox_inches='tight', transparent=True)
    plt.savefig(os.path.join(d, f'{name}.png'), bbox_inches='tight', transparent=True, dpi=180)
    if show:
        plt.show()
    plt.close()


apply()
# the chapter scripts call the style as st.<name> (import sfm_style as st): the same names here
import types
st = types.SimpleNamespace(COL=COL, PALETTE=PALETTE, MainBlue=MainBlue, IDAred=IDAred, Forest=Forest, Amber=Amber,
                           Orange=Orange, Purple=Purple, Teal=Teal, Crimson=Crimson, DarkText=DarkText, apply=apply, legend_outside_bottom=legend_outside_bottom,
                           fig_legend_bottom=fig_legend_bottom, save_fig=save_fig, check_no_grey=check_no_grey)

## Data

- Daily market data from EODHD, saved in `data/market` of the SFM repository (read locally, or from GitHub in Colab); EUR/RON is the official BNR reference rate.
- Returns in %: $r_t = 100(\ln P_t - \ln P_{t-1})$; each series on its own calendar.

In [ ]:
REPO_RAW = 'https://raw.githubusercontent.com/danpele/SFM/main/data/market/'
# local copy of the course data, if the notebook runs inside the repository
MARKET_DIR = next((os.path.join(d, 'data', 'market') for d in ('.', '..', '../..', '../../..')
                   if os.path.isdir(os.path.join(d, 'data', 'market'))), '')
END = '2026-09-18'          # last day of the saved data

# name -> (symbol, label, group, field, default start)
SERIES = {
    # equity indices
    'sp500':    ('GSPC.INDX',     'S&P 500',               'Equity', 'close', '2000-01-01'),
    'ndx':      ('NDX.INDX',      'Nasdaq 100',            'Equity', 'close', '2000-01-01'),
    'dax':      ('GDAXI.INDX',    'DAX',                   'Equity', 'close', '2000-01-01'),
    'stoxx50':  ('STOXX50E.INDX', 'Euro Stoxx 50',         'Equity', 'close', '2000-01-01'),
    'nikkei':   ('N225.INDX',     'Nikkei 225',            'Equity', 'close', '2000-01-01'),
    'bet':      ('BET',           'BET',                   'Equity', 'close', '2000-01-01'),
    'bettr':    ('BETTR.INDX',    'BET-TR',                'Equity', 'close', '2014-09-23'),
    'betxt':    ('BETXT.INDX',    'BET-XT',                'Equity', 'close', '2011-08-12'),
    'betfi':    ('BETFI.INDX',    'BET-FI',                'Equity', 'close', '2012-01-25'),
    'wig20':    ('WIG20.INDX',    'WIG20',                 'Equity', 'close', '2000-01-01'),
    'bux':      ('BUX.INDX',      'BUX',                   'Equity', 'close', '2000-01-01'),
    'px':       ('PX.INDX',       'PX',                    'Equity', 'close', '2000-01-01'),
    'vix':      ('VIX.INDX',      'VIX',                   'Volatility', 'close', '2000-01-01'),
    # ETFs (adjusted close)
    'spy':      ('SPY.US',        'SPY',                   'ETF', 'adjusted_close', '2000-01-01'),
    'tlt':      ('TLT.US',        'TLT (US Treasuries 20y+)', 'ETF', 'adjusted_close', '2002-07-30'),
    'gld':      ('GLD.US',        'GLD (gold ETF)',        'ETF', 'adjusted_close', '2004-11-18'),
    'tvbetetf': ('TVBETETF.RO',   'TVBETETF',              'ETF', 'adjusted_close', '2015-06-12'),
    # Bucharest Stock Exchange (adjusted close)
    'tlv':      ('TLV.RO',        'Banca Transilvania',    'Stock', 'adjusted_close', '2000-01-01'),
    'snp':      ('SNP.RO',        'OMV Petrom',            'Stock', 'adjusted_close', '2001-09-04'),
    'brd':      ('BRD.RO',        'BRD',                   'Stock', 'adjusted_close', '2001-01-16'),
    'tgn':      ('TGN.RO',        'Transgaz',              'Stock', 'adjusted_close', '2008-01-25'),
    'sng':      ('SNG.RO',        'Romgaz',                'Stock', 'adjusted_close', '2013-11-12'),
    'snn':      ('SNN.RO',        'Nuclearelectrica',      'Stock', 'adjusted_close', '2013-11-04'),
    'h2o':      ('H2O.RO',        'Hidroelectrica',        'Stock', 'adjusted_close', '2023-07-12'),
    # US stocks (adjusted close)
    'aapl':     ('AAPL.US',       'Apple',                 'Stock', 'adjusted_close', '2000-01-01'),
    'msft':     ('MSFT.US',       'Microsoft',             'Stock', 'adjusted_close', '2000-01-01'),
    'nvda':     ('NVDA.US',       'NVIDIA',                'Stock', 'adjusted_close', '2000-01-01'),
    'jpm':      ('JPM.US',        'JPMorgan Chase',        'Stock', 'adjusted_close', '2000-01-01'),
    # crypto (close, 7 days a week)
    'btc':      ('BTC-USD.CC',    'Bitcoin',               'Crypto', 'close', '2014-09-17'),
    'eth':      ('ETH-USD.CC',    'Ethereum',              'Crypto', 'close', '2015-08-07'),
    'sol':      ('SOL-USD.CC',    'Solana',                'Crypto', 'close', '2020-04-11'),
    'usdt':     ('USDT-USD.CC',   'Tether (USDT)',         'Crypto', 'close', '2015-02-26'),
    # FX and commodities
    'eurron':   ('REF:EUR',       'EUR/RON (BNR reference)', 'FX', 'close', '2005-07-01'),
    'usdron':   ('REF:USD',       'USD/RON (BNR reference)', 'FX', 'close', '2005-07-01'),
    'eurron_eodhd': ('EURRON.FOREX', 'EUR/RON (EODHD)',    'FX', 'close', '2005-07-01'),
    'eurusd':   ('EURUSD.FOREX',  'EUR/USD',               'FX', 'close', '2002-05-06'),
    'gold':     ('XAUUSD.FOREX',  'Gold (XAU/USD)',        'Commodity', 'close', '2000-01-01'),
    # government bond yields (close, % p.a.)
    'us10y':    ('US10Y.GBOND',   'US 10Y yield',          'Yield', 'close', '2000-01-01'),
    'de10y':    ('DE10Y.GBOND',   'Germany 10Y yield',     'Yield', 'close', '2007-03-28'),
    'ro10y':    ('RO10Y.GBOND',   'Romania 10Y yield',     'Yield', 'close', '2007-08-17'),
}
LABELS = {k: v[1] for k, v in SERIES.items()}
_CACHE = {}


def read_market(symbol):
    """Daily table of one symbol: local copy of the course data, otherwise the SFM repository on GitHub."""
    fname = f'{symbol}.csv'
    path = os.path.join(MARKET_DIR, fname) if MARKET_DIR else ''
    src = path if path and os.path.exists(path) else REPO_RAW + fname
    return pd.read_csv(src, index_col='date', parse_dates=True).sort_index()


def read_reference_rate(currency='EUR', start='2005-07-01', end=END):
    """Official BNR reference rate (RON per unit of currency), from the public yearly XML archives."""
    key = (currency, start, end)
    if key in _CACHE:
        return _CACHE[key]
    rows = []
    for y in range(max(int(start[:4]), 2005), int(end[:4]) + 1):
        url = f'https://curs.bnr.ro/files/xml/years/nbrfxrates{y}.xml'
        xml = urllib.request.urlopen(urllib.request.Request(url, headers={'User-Agent': 'Mozilla'}),
                                     timeout=60).read().decode()
        for d, body in re.findall(r'<Cube date="([\d-]+)">(.*?)</Cube>', xml, re.S):
            m = re.search(rf'<Rate currency="{currency}"(?: multiplier="\d+")?>([\d.]+)</Rate>', body)
            if m:
                rows.append((d, float(m.group(1))))
    s = pd.DataFrame(rows, columns=['date', 'close']).drop_duplicates('date').set_index('date')['close']
    s.index = pd.to_datetime(s.index)
    _CACHE[key] = s.sort_index().loc[start:end]
    return _CACHE[key]


def load_close(name, start=None, end=END, field=None):
    """Daily closing price of a named series, cleaned with the course conventions."""
    symbol, label, group, field0, start0 = SERIES[name]
    start, field = start or start0, field or field0
    if symbol.startswith('REF:'):
        return read_reference_rate(symbol.split(':')[1], start=start, end=end).rename(name)
    t = read_market(symbol)
    s = (t[field] if field in t.columns else t['close']).loc[start:end]
    s = pd.to_numeric(s, errors='coerce').dropna()
    if group != 'Yield':
        s = s[s > 0]
    if group != 'Crypto':
        s = s[s.index.dayofweek < 5]          # no weekend quotes
        s = s[s.diff() != 0]                  # no holidays filled with the previous price
    s.index.name = 'date'
    return s.rename(name)


def load_ohlc(name, start=None, end=END):
    """Open, high, low, close (and volume) of a series with OHLC data (range-based volatility estimators)."""
    symbol, _, group, _, start0 = SERIES[name]
    t = read_market(symbol).loc[start or start0:end]
    cols = [c for c in ('open', 'high', 'low', 'close', 'volume') if c in t.columns]
    t = t[cols].dropna()
    if group != 'Crypto':
        t = t[t.index.dayofweek < 5]
    return t[(t[['open', 'high', 'low', 'close']] > 0).all(axis=1)] if 'open' in cols else t


def log_returns(name, start=None, end=END, pct=True):
    """Daily log returns r_t = ln P_t - ln P_{t-1} (in % by default), on the series' own calendar."""
    r = np.log(load_close(name, start, end)).diff().dropna()
    return (100 * r if pct else r).rename(name)


def simple_returns(name, start=None, end=END, pct=True):
    """Daily simple returns R_t = P_t / P_{t-1} - 1 (in % by default)."""
    r = load_close(name, start, end).pct_change().dropna()
    return (100 * r if pct else r).rename(name)


def load_panel(names, start=None, end=END, kind='close'):
    """Several series aligned on date: kind = 'close' or 'returns' (NaN where a market does not trade)."""
    f = load_close if kind == 'close' else log_returns
    return pd.concat([f(n, start, end) for n in names], axis=1)


def periods_per_year(r):
    """Average number of observations per calendar year (the actual frequency of the series)."""
    years = (r.index[-1] - r.index[0]).days / 365.25
    return len(r) / years

## Seminar functions

- The candidate models: `fit_model`, `logpdf`, `cdf`, `ppf`, `var_es` (Normal, Student-t, skewed-t, GED, Normal mixture, NIG, stable).
- Tests and scores: `lr_test`, `vuong`, `edf_stats` (KS, CvM, AD), `lilliefors_crit`, `kupiec`, `pinball`.
- `ic_table` (AIC, BIC, Δ, Akaike weights) and `fit_table` (fit several models and tabulate them).

In [ ]:
from scipy import optimize
from scipy.special import gammaln
START = '2000-01-01'
STOCK_START = '2010-01-01'
ASSETS = ['bet', 'sp500', 'dax', 'btc']
STOCKS = ['tlv', 'snp', 'brd']
BAD_DAYS = {'tlv': ['2016-05-30', '2016-05-31']}
MODELS = ['Normal', 'Student-t', 'Skewed-t', 'GED', 'Mixture', 'NIG', 'Stable']
K = {'Normal': 2, 'Student-t': 3, 'Skewed-t': 4, 'GED': 3, 'Mixture': 5, 'NIG': 4, 'Stable': 4}
MODEL_COL = {'Normal': '#CD0000', 'Student-t': '#2E7D32', 'Skewed-t': '#8E44AD', 'GED': '#E67E22', 'Mixture': '#17A2B8', 'NIG': '#B5853F', 'Stable': '#1A3A6E', 'Data': 'black', 'Historical': '#DC3545'}
SHORT = {'bet': 'BET', 'sp500': 'S&P 500', 'dax': 'DAX', 'btc': 'Bitcoin', 'tlv': 'TLV', 'snp': 'SNP', 'brd': 'BRD'}
COLORS = {'sp500': '#1A3A6E', 'dax': '#17A2B8', 'bet': '#CD0000', 'btc': '#B5853F', 'tlv': '#E67E22', 'snp': '#2E7D32', 'brd': '#8E44AD'}
SEED = 2026
SPLIT = '2019-12-31'
ALPHA_VAR = 0.01
PANEL = ['Normal', 'Student-t', 'Skewed-t', 'Stable']
ALPHA_ES = 0.025
BLOCK = 20


def stable_cf(t, alpha, beta, gamma=1.0, delta=0.0, param=0):
    """Characteristic function E exp(itX) of S(alpha, beta, gamma, delta; param), Nolan's S0 (param=0) or S1."""
    t = np.asarray(t, float)
    a = np.abs(gamma * t)
    if alpha != 1:
        tan = np.tan(np.pi * alpha / 2)
        if param == 0:      # -|gt|^a [1 + i b tan(pi a/2) sign(t) (|gt|^(1-a) - 1)], written without 0^(1-a)
            psi = -a ** alpha - 1j * beta * tan * (gamma * t - np.sign(t) * a ** alpha)
        else:               # -|gt|^a [1 - i b tan(pi a/2) sign(t)]
            psi = -a ** alpha * (1 - 1j * beta * tan * np.sign(t))
    else:
        arg = a if param == 0 else np.abs(t)
        lg = np.log(np.where(arg > 0, arg, 1.0))
        psi = -a * (1 + 1j * beta * (2 / np.pi) * np.sign(t) * lg)
    return np.exp(psi + 1j * delta * t)


def s0_to_s1(alpha, beta, gamma, delta0):
    """Location in S1 from the location in S0."""
    if alpha != 1:
        return delta0 - beta * gamma * np.tan(np.pi * alpha / 2)
    return delta0 - beta * (2 / np.pi) * gamma * np.log(gamma)


class StableGrid:
    """Density and distribution function of the standard S0(alpha, beta, 1, 0) law on a fine grid, by FFT
    inversion of the characteristic function, f(x) = (1/2pi) int phi(t) exp(-itx) dt (Mittnik et al., 1999).
    Grid step h = 0.01 on [-327.68, 327.67]; the power-law tail approximation is used beyond the grid."""

    def __init__(self, alpha, beta, h=0.01, N=2 ** 16):
        k = np.fft.fftfreq(N, d=1.0 / N)                        # 0, 1, ..., N/2-1, -N/2, ..., -1
        dt = 2 * np.pi / (N * h)
        phi = stable_cf(k * dt, alpha, beta, 1.0, 0.0, 0) * np.where(k % 2 == 0, 1.0, -1.0)
        f = np.real(np.fft.fft(phi)) * dt / (2 * np.pi)
        self.alpha, self.beta, self.h = alpha, beta, h
        self.x = (np.arange(N) - N / 2) * h
        self.f = np.maximum(f, 1e-300)
        self.logf = np.log(self.f)
        # tail constant: P(X > x) ~ c_alpha (1 + beta) x^(-alpha), c_alpha = Gamma(alpha) sin(pi alpha / 2) / pi
        from scipy.special import gamma as G
        self.c = 0.0 if alpha >= 2 else G(alpha) * np.sin(np.pi * alpha / 2) / np.pi
        left = self.c * (1 - beta) * abs(self.x[0]) ** (-alpha) if alpha < 2 else 0.0
        right = self.c * (1 + beta) * self.x[-1] ** (-alpha) if alpha < 2 else 0.0
        cells = 0.5 * (self.f[1:] + self.f[:-1]) * h
        self.F = np.clip(left + np.concatenate([[0.0], np.cumsum(cells)]), 0.0, 1.0)           # P(X <= x), from the left
        self.S = np.clip(right + np.concatenate([np.cumsum(cells[::-1])[::-1], [0.0]]), 0.0, 1.0)  # P(X > x), from the right

    def pdf(self, z):
        z = np.asarray(z, float)
        out = np.exp(np.interp(z, self.x, self.logf))
        far = np.abs(z) > self.x[-1]
        if far.any() and self.alpha < 2:
            out[far] = self.alpha * self.c * (1 + np.sign(z[far]) * self.beta) * np.abs(z[far]) ** (-self.alpha - 1)
        return out

    def cdf(self, z):
        z = np.asarray(z, float)
        return np.where(z <= 0, np.interp(z, self.x, self.F), 1 - np.interp(z, self.x, self.S))

    def sf(self, z):
        return 1 - self.cdf(z) if np.ndim(z) == 0 else np.where(np.asarray(z) > 0, np.interp(z, self.x, self.S), 1 - self.cdf(z))

    def ppf(self, p):
        i = np.searchsorted(self.F, 1e-12) if self.F[0] < 1e-12 else 0
        return np.interp(p, self.F[i:], self.x[i:])


def mcculloch(x):
    """McCulloch (1986) quantile estimator of (alpha, beta, gamma, delta0, delta1) from a sample;
    sample quantiles: x_(i) is the (2i - 1)/(2n) quantile, linear interpolation."""
    q05, q25, q50, q75, q95 = np.percentile(np.asarray(x, float), [5, 25, 50, 75, 95], method='hazen')
    return mcculloch_quantiles(q05, q25, q50, q75, q95)


def mcculloch_quantiles(q05, q25, q50, q75, q95):
    """McCulloch (1986) estimates from the 5%, 25%, 50%, 75% and 95% quantiles.
    nu_alpha = (q95 - q05)/(q75 - q25), nu_beta = (q95 + q05 - 2 q50)/(q95 - q05); alpha and beta from
    Tables III-IV, gamma = (q75 - q25)/phi_3(alpha, beta) (Table V), the S0 location from Table VII."""
    from scipy.interpolate import RegularGridInterpolator as RGI
    nu_a_grid = [2.439, 2.5, 2.6, 2.7, 2.8, 3.0, 3.2, 3.5, 4.0, 5.0, 6.0, 8.0, 10.0, 15.0, 25.0]
    nu_b_grid = [0.0, 0.1, 0.2, 0.3, 0.5, 0.7, 1.0]
    T3 = [[2.000, 2.000, 2.000, 2.000, 2.000, 2.000, 2.000], [1.916, 1.924, 1.924, 1.924, 1.924, 1.924, 1.924],
          [1.808, 1.813, 1.829, 1.829, 1.829, 1.829, 1.829], [1.729, 1.730, 1.737, 1.745, 1.745, 1.745, 1.745],
          [1.664, 1.663, 1.663, 1.668, 1.676, 1.676, 1.676], [1.563, 1.560, 1.553, 1.548, 1.547, 1.547, 1.547],
          [1.484, 1.480, 1.471, 1.460, 1.448, 1.438, 1.438], [1.391, 1.386, 1.378, 1.364, 1.337, 1.318, 1.318],
          [1.279, 1.273, 1.266, 1.250, 1.210, 1.184, 1.150], [1.128, 1.121, 1.114, 1.101, 1.067, 1.027, 0.973],
          [1.029, 1.021, 1.014, 1.004, 0.974, 0.935, 0.874], [0.896, 0.892, 0.884, 0.883, 0.855, 0.823, 0.769],
          [0.818, 0.812, 0.806, 0.801, 0.780, 0.756, 0.691], [0.698, 0.695, 0.692, 0.689, 0.676, 0.656, 0.597],
          [0.593, 0.590, 0.588, 0.586, 0.579, 0.563, 0.513]]
    T4 = [[0, 2.160, 1.000, 1.000, 1.000, 1.000, 1.000], [0, 1.592, 3.390, 1.000, 1.000, 1.000, 1.000],
          [0, 0.759, 1.800, 1.000, 1.000, 1.000, 1.000], [0, 0.482, 1.048, 1.694, 1.000, 1.000, 1.000],
          [0, 0.360, 0.760, 1.232, 2.229, 1.000, 1.000], [0, 0.253, 0.518, 0.823, 1.575, 1.000, 1.000],
          [0, 0.203, 0.410, 0.632, 1.244, 1.906, 1.000], [0, 0.165, 0.332, 0.499, 0.943, 1.560, 1.000],
          [0, 0.136, 0.271, 0.404, 0.689, 1.230, 2.195], [0, 0.109, 0.216, 0.323, 0.539, 0.827, 1.917],
          [0, 0.096, 0.190, 0.284, 0.472, 0.693, 1.759], [0, 0.082, 0.163, 0.243, 0.412, 0.601, 1.596],
          [0, 0.074, 0.147, 0.220, 0.377, 0.546, 1.482], [0, 0.064, 0.128, 0.191, 0.330, 0.478, 1.362],
          [0, 0.056, 0.112, 0.167, 0.285, 0.428, 1.274]]
    a_grid = [0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.3, 1.4, 1.5, 1.6, 1.7, 1.8, 1.9, 2.0]
    b_grid = [0.0, 0.25, 0.5, 0.75, 1.0]
    T5 = [[2.588, 3.073, 4.534, 6.636, 9.144], [2.337, 2.634, 3.542, 4.808, 6.247], [2.189, 2.392, 3.004, 3.844, 4.775],
          [2.098, 2.244, 2.676, 3.265, 3.912], [2.040, 2.149, 2.461, 2.886, 3.356], [2.000, 2.085, 2.311, 2.624, 2.973],
          [1.980, 2.040, 2.205, 2.435, 2.696], [1.965, 2.007, 2.125, 2.294, 2.491], [1.955, 1.984, 2.067, 2.188, 2.333],
          [1.946, 1.967, 2.022, 2.106, 2.211], [1.939, 1.952, 1.988, 2.045, 2.116], [1.933, 1.940, 1.962, 1.997, 2.043],
          [1.927, 1.930, 1.943, 1.961, 1.987], [1.921, 1.922, 1.927, 1.936, 1.947], [1.914, 1.915, 1.916, 1.918, 1.921],
          [1.908, 1.908, 1.908, 1.908, 1.908]]
    T7 = [[0, -0.061, -0.279, -0.659, -1.198], [0, -0.078, -0.272, -0.581, -0.997], [0, -0.089, -0.262, -0.520, -0.853],
          [0, -0.096, -0.250, -0.469, -0.742], [0, -0.099, -0.237, -0.424, -0.652], [0, -0.098, -0.223, -0.380, -0.576],
          [0, -0.095, -0.208, -0.346, -0.508], [0, -0.090, -0.192, -0.310, -0.447], [0, -0.084, -0.173, -0.276, -0.390],
          [0, -0.075, -0.154, -0.241, -0.335], [0, -0.066, -0.134, -0.206, -0.283], [0, -0.056, -0.111, -0.170, -0.232],
          [0, -0.043, -0.088, -0.132, -0.179], [0, -0.030, -0.061, -0.092, -0.123], [0, -0.017, -0.032, -0.049, -0.064],
          [0, 0.000, 0.000, 0.000, 0.000]]
    psi1 = RGI((nu_a_grid, nu_b_grid), np.array(T3), bounds_error=False, fill_value=None)
    psi2 = RGI((nu_a_grid, nu_b_grid), np.array(T4), bounds_error=False, fill_value=None)
    phi3 = RGI((a_grid, b_grid), np.array(T5), bounds_error=False, fill_value=None)
    phi5 = RGI((a_grid, b_grid), np.array(T7), bounds_error=False, fill_value=None)
    nu_a = (q95 - q05) / (q75 - q25)
    nu_b = (q95 + q05 - 2 * q50) / (q95 - q05)
    if nu_a >= 2.439:
        alpha = float(np.clip(psi1([[nu_a, abs(nu_b)]])[0], 0.5, 2.0))
        beta = float(np.clip(np.sign(nu_b) * psi2([[nu_a, abs(nu_b)]])[0], -1.0, 1.0))
    else:
        alpha, beta = 2.0, 0.0
    gamma = float((q75 - q25) / phi3([[alpha, abs(beta)]])[0])
    delta0 = float(q50 + gamma * np.sign(beta) * phi5([[alpha, abs(beta)]])[0])   # zeta in McCulloch (1986)
    delta1 = float(s0_to_s1(alpha, beta, gamma, delta0)) if alpha != 1 else delta0
    return {'alpha': alpha, 'beta': beta, 'gamma': gamma, 'delta0': delta0, 'delta1': delta1,
            'nu_alpha': float(nu_a), 'nu_beta': float(nu_b),
            'q05': float(q05), 'q25': float(q25), 'q50': float(q50), 'q75': float(q75), 'q95': float(q95)}


def stable_nll(p, x):
    """Negative log-likelihood of S0(alpha, beta, gamma, delta0) at p = (alpha, beta, log gamma, delta0)."""
    a, b, lg, d = p
    if not (1.01 <= a <= 1.999 and -0.999 <= b <= 0.999):
        return 1e12
    G = StableGrid(a, b)
    return -np.sum(np.log(G.pdf((x - d) / np.exp(lg)))) + len(x) * lg


def stable_mle(x, start=None, se=True):
    """Maximum likelihood estimate of (alpha, beta, gamma, delta0) in S0 (Nelder-Mead from the McCulloch estimate),
    with standard errors from the numerical Hessian of the log-likelihood; delta1 is the S1 location."""
    from scipy.optimize import minimize
    x = np.asarray(x, float)
    m = start or mcculloch(x)
    p0 = [min(max(m['alpha'], 1.05), 1.98), float(np.clip(m['beta'], -0.95, 0.95)), np.log(m['gamma']), m['delta0']]
    res = minimize(stable_nll, p0, args=(x,), method='Nelder-Mead',
                   options={'xatol': 1e-5, 'fatol': 1e-4, 'maxiter': 4000, 'maxfev': 8000})
    a, b, lg, d = res.x
    out = {'alpha': float(a), 'beta': float(b), 'gamma': float(np.exp(lg)), 'delta0': float(d),
           'delta1': float(s0_to_s1(a, b, np.exp(lg), d)), 'loglik': float(-res.fun), 'n': len(x)}
    if se:
        th = np.array([a, b, np.exp(lg), d])

        def f(v):
            return stable_nll([v[0], v[1], np.log(v[2]), v[3]], x)
        steps = np.array([1e-3, 1e-2, 1e-3 * th[2], 1e-3 * th[2]])
        H = np.zeros((4, 4))
        for i in range(4):
            for j in range(i, 4):
                ei, ej = np.eye(4)[i] * steps[i], np.eye(4)[j] * steps[j]
                H[i, j] = H[j, i] = (f(th + ei + ej) - f(th + ei - ej) - f(th - ei + ej) + f(th - ei - ej)) / (4 * steps[i] * steps[j])
        try:
            cov = np.linalg.inv(H)
            sd = np.sqrt(np.clip(np.diag(cov), 0, None))
        except np.linalg.LinAlgError:
            sd = np.full(4, np.nan)
        out.update({'se_alpha': float(sd[0]), 'se_beta': float(sd[1]), 'se_gamma': float(sd[2]), 'se_delta0': float(sd[3])})
    return out


def returns(k, start=None, end=None):
    """Daily log returns in %: indices since 2000, Bitcoin over its full history, Bucharest stocks since 2010
    (the two erroneous Banca Transilvania days of May 2016 removed)."""
    s0 = start or (STOCK_START if k in STOCKS else START if k != 'btc' else None)
    r = log_returns(k, s0)
    if end:
        r = r.loc[:end]
    if k in BAD_DAYS:
        r = r.drop(pd.to_datetime(BAD_DAYS[k]), errors='ignore')
    return r


def skewt_consts(nu, lam):
    """Constants a, b, c of the skewed Student-t of Hansen (1994), standardised to mean 0 and variance 1."""
    c = np.exp(gammaln((nu + 1) / 2) - gammaln(nu / 2)) / np.sqrt(np.pi * (nu - 2))
    a = 4 * lam * c * (nu - 2) / (nu - 1)
    b = np.sqrt(1 + 3 * lam ** 2 - a ** 2)
    return a, b, c


def skewt_logpdf(x, mu, sigma, nu, lam):
    """Log-density of Hansen's skewed-t with mean mu, standard deviation sigma, degrees of freedom nu > 2 and
    skewness lam in (-1, 1); lam < 0: longer left tail; lam = 0: the Student-t with variance sigma^2."""
    a, b, c = skewt_consts(nu, lam)
    z = (np.asarray(x, float) - mu) / sigma
    s = np.where(z < -a / b, 1 - lam, 1 + lam)
    return np.log(b * c / sigma) - (nu + 1) / 2 * np.log1p(((b * z + a) / s) ** 2 / (nu - 2))


def skewt_cdf(x, mu, sigma, nu, lam):
    a, b, c = skewt_consts(nu, lam)
    z = (np.asarray(x, float) - mu) / sigma
    k = np.sqrt(nu / (nu - 2))
    lo = (1 - lam) * stats.t.cdf(k * (b * z + a) / (1 - lam), nu)
    hi = (1 - lam) / 2 + (1 + lam) * (stats.t.cdf(k * (b * z + a) / (1 + lam), nu) - 0.5)
    return np.where(z < -a / b, lo, hi)


def skewt_ppf(p, mu, sigma, nu, lam):
    a, b, c = skewt_consts(nu, lam)
    p = np.asarray(p, float)
    k = np.sqrt((nu - 2) / nu)
    lo = ((1 - lam) * k * stats.t.ppf(np.clip(p / (1 - lam), 1e-300, 1), nu) - a) / b
    hi = ((1 + lam) * k * stats.t.ppf(np.clip(0.5 + (p - (1 - lam) / 2) / (1 + lam), 0, 1 - 1e-16), nu) - a) / b
    return mu + sigma * np.where(p < (1 - lam) / 2, lo, hi)


def mixture_em(x, k=2, starts=4, iters=600, tol=1e-9, seed=SEED):
    """Maximum likelihood fit of a k-component Normal mixture by the EM algorithm (several starts, the best kept)."""
    x = np.asarray(x, float)
    rng = np.random.default_rng(seed)
    best = None
    for s in range(starts):
        if s == 0:      # start: equal weights, means at quantiles, spread scales
            w = np.full(k, 1 / k)
            mu = np.quantile(x, (np.arange(k) + 0.5) / k) * 0.2
            sd = x.std() * np.linspace(0.6, 2.0, k)
        else:
            w = rng.dirichlet(np.ones(k) * 3)
            mu = rng.choice(x, k) * 0.3
            sd = x.std() * rng.uniform(0.4, 2.5, k)
        ll_old = -np.inf
        for _ in range(iters):
            lp = np.log(w) + stats.norm.logpdf(x[:, None], mu, sd)
            m = lp.max(axis=1, keepdims=True)
            lse = m[:, 0] + np.log(np.exp(lp - m).sum(axis=1))
            ll = lse.sum()
            g = np.exp(lp - lse[:, None])
            nk = g.sum(axis=0) + 1e-12
            w = nk / len(x)
            mu = (g * x[:, None]).sum(axis=0) / nk
            sd = np.sqrt((g * (x[:, None] - mu) ** 2).sum(axis=0) / nk)
            sd = np.maximum(sd, 1e-3 * x.std())
            if abs(ll - ll_old) < tol * abs(ll):
                break
            ll_old = ll
        if best is None or ll > best['loglik']:
            o = np.argsort(sd)
            best = {'w': w[o].tolist(), 'mu': mu[o].tolist(), 'sd': sd[o].tolist(), 'loglik': float(ll)}
    return best


def mixture_logpdf(x, w, mu, sd):
    lp = np.log(np.asarray(w)) + stats.norm.logpdf(np.asarray(x, float)[:, None], mu, sd)
    m = lp.max(axis=1, keepdims=True)
    return m[:, 0] + np.log(np.exp(lp - m).sum(axis=1))


def mixture_cdf(x, w, mu, sd):
    return (np.asarray(w) * stats.norm.cdf(np.asarray(x, float)[:, None], mu, sd)).sum(axis=1)


def mixture_ppf(p, w, mu, sd):
    p = np.atleast_1d(np.asarray(p, float))
    lo, hi = min(mu) - 60 * max(sd), max(mu) + 60 * max(sd)
    return np.array([optimize.brentq(lambda z: mixture_cdf([z], w, mu, sd)[0] - q, lo, hi) for q in p])


def nig_fit(x):
    """Maximum likelihood fit of the NIG law (scipy.stats.norminvgauss: tail a > 0, skewness |b| < a, loc, scale)."""
    x = np.asarray(x, float)

    def nll(p):
        a = np.exp(p[0])
        b = a * np.tanh(p[1])
        v = -np.sum(stats.norminvgauss.logpdf(x, a, b, p[2], np.exp(p[3])))
        return v if np.isfinite(v) else 1e12
    sd = x.std()
    p0 = [np.log(1.0), 0.0, np.median(x), np.log(sd)]
    res = optimize.minimize(nll, p0, method='Nelder-Mead', options={'xatol': 1e-6, 'fatol': 1e-6, 'maxiter': 6000, 'maxfev': 12000})
    a = float(np.exp(res.x[0]))
    return {'a': a, 'b': float(a * np.tanh(res.x[1])), 'loc': float(res.x[2]), 'scale': float(np.exp(res.x[3])), 'loglik': float(-res.fun)}


def skewt_fit(x, start=None):
    """Maximum likelihood fit of Hansen's skewed-t (mu, sigma, nu, lam)."""
    x = np.asarray(x, float)

    def nll(p):
        nu, lam = 2 + np.exp(p[2]), np.tanh(p[3])
        if nu - 2 < 1e-6 or abs(lam) > 0.999:
            return 1e12
        v = -np.sum(skewt_logpdf(x, p[0], np.exp(p[1]), nu, lam))
        return v if np.isfinite(v) else 1e12
    if start is None:
        nu0 = stats.t.fit(x)[0] if len(x) < 3000 else 4.0
        start = [x.mean(), np.log(x.std()), np.log(max(nu0 - 2, 0.5)), 0.0]
    res = optimize.minimize(nll, start, method='Nelder-Mead', options={'xatol': 1e-7, 'fatol': 1e-7, 'maxiter': 8000, 'maxfev': 16000})
    return {'mu': float(res.x[0]), 'sigma': float(np.exp(res.x[1])), 'nu': float(2 + np.exp(res.x[2])),
            'lam': float(np.tanh(res.x[3])), 'loglik': float(-res.fun)}


def ged_fit(x):
    """Maximum likelihood fit of the GED (scipy.stats.gennorm: shape beta, loc, scale); beta = 2 is the Normal law,
    beta = 1 the Laplace law, beta < 2 heavier tails than the Normal law."""
    x = np.asarray(x, float)

    def nll(p):
        v = -np.sum(stats.gennorm.logpdf(x, np.exp(p[0]), p[1], np.exp(p[2])))
        return v if np.isfinite(v) else 1e12
    res = optimize.minimize(nll, [np.log(1.2), np.median(x), np.log(x.std())], method='Nelder-Mead',
                            options={'xatol': 1e-7, 'fatol': 1e-7, 'maxiter': 4000})
    return {'beta': float(np.exp(res.x[0])), 'loc': float(res.x[1]), 'scale': float(np.exp(res.x[2])), 'loglik': float(-res.fun)}


def t_fit(x):
    """Maximum likelihood fit of the location-scale Student-t (nu, loc, scale)."""
    x = np.asarray(x, float)
    nu, loc, sc = stats.t.fit(x)
    return {'nu': float(nu), 'loc': float(loc), 'scale': float(sc), 'loglik': float(np.sum(stats.t.logpdf(x, nu, loc, sc)))}


def fit_model(name, x):
    """Maximum likelihood fit of one candidate model; returns its parameters and the maximised log-likelihood."""
    x = np.asarray(x, float)
    if name == 'Normal':
        mu, sd = x.mean(), x.std(ddof=0)
        return {'mu': float(mu), 'sigma': float(sd), 'loglik': float(np.sum(stats.norm.logpdf(x, mu, sd)))}
    if name == 'Student-t':
        return t_fit(x)
    if name == 'Skewed-t':
        return skewt_fit(x)
    if name == 'GED':
        return ged_fit(x)
    if name == 'Mixture':
        return mixture_em(x, 2)
    if name == 'NIG':
        return nig_fit(x)
    if name == 'Stable':
        s = stable_mle(x, se=False)
        return dict(s, loglik=float(s['loglik']))
    raise ValueError(name)


def logpdf(name, x, p):
    x = np.asarray(x, float)
    if name == 'Normal':
        return stats.norm.logpdf(x, p['mu'], p['sigma'])
    if name == 'Student-t':
        return stats.t.logpdf(x, p['nu'], p['loc'], p['scale'])
    if name == 'Skewed-t':
        return skewt_logpdf(x, p['mu'], p['sigma'], p['nu'], p['lam'])
    if name == 'GED':
        return stats.gennorm.logpdf(x, p['beta'], p['loc'], p['scale'])
    if name == 'Mixture':
        return mixture_logpdf(x, p['w'], p['mu'], p['sd'])
    if name == 'NIG':
        return stats.norminvgauss.logpdf(x, p['a'], p['b'], p['loc'], p['scale'])
    if name == 'Stable':
        return np.log(StableGrid(p['alpha'], p['beta']).pdf((x - p['delta0']) / p['gamma']) / p['gamma'])
    raise ValueError(name)


def cdf(name, x, p):
    x = np.asarray(x, float)
    if name == 'Normal':
        return stats.norm.cdf(x, p['mu'], p['sigma'])
    if name == 'Student-t':
        return stats.t.cdf(x, p['nu'], p['loc'], p['scale'])
    if name == 'Skewed-t':
        return skewt_cdf(x, p['mu'], p['sigma'], p['nu'], p['lam'])
    if name == 'GED':
        return stats.gennorm.cdf(x, p['beta'], p['loc'], p['scale'])
    if name == 'Mixture':
        return mixture_cdf(x, p['w'], p['mu'], p['sd'])
    if name == 'NIG':
        return stats.norminvgauss.cdf(x, p['a'], p['b'], p['loc'], p['scale'])
    if name == 'Stable':
        return StableGrid(p['alpha'], p['beta']).cdf((x - p['delta0']) / p['gamma'])
    raise ValueError(name)


def ppf(name, q, p):
    q = np.asarray(q, float)
    if name == 'Normal':
        return stats.norm.ppf(q, p['mu'], p['sigma'])
    if name == 'Student-t':
        return stats.t.ppf(q, p['nu'], p['loc'], p['scale'])
    if name == 'Skewed-t':
        return skewt_ppf(q, p['mu'], p['sigma'], p['nu'], p['lam'])
    if name == 'GED':
        return stats.gennorm.ppf(q, p['beta'], p['loc'], p['scale'])
    if name == 'Mixture':
        return mixture_ppf(q, p['w'], p['mu'], p['sd'])
    if name == 'NIG':
        return stats.norminvgauss.ppf(q, p['a'], p['b'], p['loc'], p['scale'])
    if name == 'Stable':
        return p['delta0'] + p['gamma'] * StableGrid(p['alpha'], p['beta']).ppf(q)
    raise ValueError(name)


def var_es(name, p, a_var=ALPHA_VAR, a_es=ALPHA_ES, m=4000):
    """VaR_a = -q_a and ES_a = -(1/a) int_0^a q_u du (midpoint rule on a fine grid of u, refined near 0)."""
    var = -float(np.ravel(ppf(name, a_var, p))[0])
    u = a_es * np.concatenate([np.logspace(-6, -2, m // 2, endpoint=False), np.linspace(0.01, 1, m // 2 + 1)])
    mid, du = 0.5 * (u[1:] + u[:-1]), np.diff(u)
    qs = ppf(name, mid, p)
    es = -float(np.sum(qs * du) + np.ravel(ppf(name, u[0] / 2, p))[0] * u[0]) / a_es
    return var, es
SEM_MODELS = ['Normal', 'Student-t', 'Skewed-t', 'GED', 'Mixture', 'NIG']


def kupiec(x, n, p=ALPHA_VAR):
    """Kupiec (1995) unconditional coverage test: LR = -2 ln[(1-p)^(n-x) p^x / ((1-x/n)^(n-x) (x/n)^x)] ~ chi2(1)."""
    phat = x / n
    l0 = (n - x) * np.log(1 - p) + x * np.log(p)
    l1 = (n - x) * np.log(1 - phat) + (x * np.log(phat) if x > 0 else 0.0)
    lr = -2 * (l0 - l1)
    return {'x': int(x), 'n': int(n), 'rate': float(phat), 'lr': float(lr), 'p': float(stats.chi2.sf(lr, 1))}


def pinball(r, q, a=ALPHA_VAR):
    """Quantile (pinball) loss of a VaR forecast q (a quantile of the return): mean of (a - 1{r < q})(r - q)."""
    r = np.asarray(r, float)
    return float(np.mean((a - (r < q)) * (r - q)))


def lr_test(ll0, ll1, df, boundary=False):
    """Likelihood-ratio test of a restricted model (ll0) inside a larger one (ll1): LR = 2 (ll1 - ll0) ~ chi2(df);
    on the boundary of the parameter space (Normal inside Student-t, 1/nu = 0) the null law is the 50:50 mixture of
    chi2(0) and chi2(1) (Self and Liang, 1987)."""
    lr = max(2 * (ll1 - ll0), 0.0)
    p = stats.chi2.sf(lr, df)
    return {'lr': float(lr), 'p': float(0.5 * p if boundary else p)}


def vuong(name1, p1, name2, p2, x):
    """Vuong (1989) test of two non-nested models: V = sqrt(n) mean(m) / sd(m), m_i = log f1(x_i) - log f2(x_i);
    V > 1.96: model 1 closer to the truth (Kullback-Leibler); V < -1.96: model 2; otherwise no decision."""
    m = logpdf(name1, x, p1) - logpdf(name2, x, p2)
    v = np.sqrt(len(m)) * m.mean() / m.std(ddof=1)
    return {'v': float(v), 'p': float(2 * stats.norm.sf(abs(v)))}


def edf_stats(u):
    """Kolmogorov-Smirnov D, Cramer-von Mises W2 and Anderson-Darling A2 from u_i = F(x_i) (F fully specified)."""
    u = np.clip(np.sort(np.asarray(u, float)), 1e-12, 1 - 1e-12)
    n = len(u)
    i = np.arange(1, n + 1)
    d = max(np.max(i / n - u), np.max(u - (i - 1) / n))
    w2 = 1 / (12 * n) + np.sum((u - (2 * i - 1) / (2 * n)) ** 2)
    a2 = -n - np.mean((2 * i - 1) * (np.log(u) + np.log(1 - u[::-1])))
    return {'ks': float(d), 'cvm': float(w2), 'ad': float(a2)}


def lilliefors_crit(n, reps=4000, seed=SEED):
    """95% critical values of KS, CvM and AD for the Normal law with estimated mean and variance (simulation)."""
    rng = np.random.default_rng(seed)
    S = {'ks': [], 'cvm': [], 'ad': []}
    for _ in range(reps):
        y = rng.standard_normal(n)
        for s, v in edf_stats(stats.norm.cdf(y, y.mean(), y.std())).items():
            S[s].append(v)
    return {s: float(np.quantile(v, 0.95)) for s, v in S.items()}


def ic_table(ll, k, n):
    """AIC, BIC, Delta AIC, Delta BIC and Akaike weights from log-likelihoods ll and numbers of parameters k."""
    aic = {m: 2 * k[m] - 2 * ll[m] for m in ll}
    bic = {m: k[m] * np.log(n) - 2 * ll[m] for m in ll}
    a0, b0 = min(aic.values()), min(bic.values())
    w = {m: np.exp(-0.5 * (aic[m] - a0)) for m in ll}
    s = sum(w.values())
    return {m: {'ll': ll[m], 'k': k[m], 'aic': aic[m], 'bic': bic[m], 'daic': aic[m] - a0, 'dbic': bic[m] - b0,
                'w': w[m] / s} for m in ll}


def fit_table(x, models=SEM_MODELS):
    """Fit the models to x; parameters, log-likelihoods and the information-criteria table."""
    P = {m: fit_model(m, x) for m in models}
    tab = ic_table({m: P[m]['loglik'] for m in models}, {m: K[m] for m in models}, len(x))
    return P, tab

# Part A: computations on paper

- Do each computation on paper first; then check it with the code.

## A1 [Solved]: four models for the S&P 500

**Context.** S&P 500, the last 1000 trading days; the maximised log-likelihoods are computed below (the slides give them rounded).

1. Compute AIC and BIC ($\ln 1000 = 6.908$) of each model.
2. Compute ΔAIC and the Akaike weights.
3. Say which model each criterion selects and how strong the evidence is.

**Report:** a table with four rows and two sentences.

In [6]:
# Solution
x = returns('sp500').values[-1000:]
ll = {m: round(fit_model(m, x)['loglik'], 1) for m in ['Normal', 'Student-t', 'Skewed-t', 'GED']}
pd.DataFrame(ic_table(ll, {m: K[m] for m in ll}, 1000)).T.round(3)

,ll,k,aic,bic,daic,dbic,w
Normal,-1400.9,2.0,2805.8,2815.616,170.4,165.492,0.000
Student-t,-1314.7,3.0,2635.4,2650.123,0.0,0.000,0.525
Skewed-t,-1313.8,4.0,2635.6,2655.231,0.2,5.108,0.475
GED,-1322.6,3.0,2651.2,2665.923,15.8,15.800,0.000


**Interpretation of the result.** AIC finds the Student-t and the skewed-t tied (Δ ≤ 2); BIC prefers the Student-t, which has one parameter less.

## A2 [Proposed]: four models for Bitcoin

**Context.** Bitcoin, the last 1000 days; models Normal, Student-t, NIG, Normal mixture. Model: A1.

1. Compute AIC, BIC, ΔAIC and the Akaike weights.
2. Say whether AIC and BIC select the same model.
3. Explain why the Normal mixture, with the most parameters, does not win.

**Report:** a table with four rows and two sentences.

In [ ]:
# Your code here


## A3 [Solved]: is skewness needed? Is the Normal law enough?

**Context.** The log-likelihoods of A1.

1. Test the Student-t ($H_0$: λ = 0) against the skewed-t with a likelihood-ratio test at 5%.
2. Test the Normal law ($H_0$: β = 2) against the GED.
3. Compare the LR decision of step 1 with the AIC verdict of A1.

**Report:** two LR values, two decisions and one sentence.

In [8]:
# Solution
x = returns('sp500').values[-1000:]
ll = {m: round(fit_model(m, x)['loglik'], 1) for m in ['Normal', 'Student-t', 'Skewed-t', 'GED']}
print('t vs skewed-t:', lr_test(ll['Student-t'], ll['Skewed-t'], 1))
print('Normal vs GED:', lr_test(ll['Normal'], ll['GED'], 1))

t vs skewed-t: {'lr': 1.800000000000182, 'p': 0.17971249487897395}
Normal vs GED: {'lr': 156.60000000000036, 'p': 6.25973453257248e-36}


**Interpretation of the result.** The skewness parameter is not significant, in line with the AIC tie; the Normal law is rejected.

## A4 [Proposed]: a test on the boundary

**Context.** The log-likelihoods of A2. Model: A3.

1. Compute LR for the Normal law inside the Student-t.
2. Explain why the null law is not χ²(1), and give the correct 5% critical value.
3. Say whether an LR test can compare the Student-t with the NIG.

**Report:** one LR value, one critical value and two sentences.

In [ ]:
# Your code here


## A5 [Solved]: KS on five returns

**Context.** Daily returns (%): −2.1, −0.4, 0.3, 0.9, 1.6.

1. Compute $D_5$ against N(0, 1) and compare it with the exact 5% critical value.
2. Estimate μ and σ (divisor n), compute $D_5$ against $N(\hat\mu, \hat\sigma^2)$ and compare it with the Lilliefors value.
3. Explain why the Lilliefors critical value is smaller.

**Report:** two values of $D_5$, two decisions, one sentence.

In [10]:
# Solution
x = np.array([-2.1, -0.4, 0.3, 0.9, 1.6])
print('known F :', round(edf_stats(stats.norm.cdf(x))['ks'], 3), ' critical', round(stats.kstwo.ppf(0.95, 5), 3))
print('estimated:', round(edf_stats(stats.norm.cdf(x, x.mean(), x.std()))['ks'], 3), ' Lilliefors', round(lilliefors_crit(5, reps=20000)['ks'], 3))

known F : 0.218  critical 0.563


estimated: 0.175  Lilliefors 0.354


**Interpretation of the result.** Neither test rejects: five observations carry almost no information about the shape.

## A6 [Proposed]: KS with one large loss

**Context.** Daily returns (%): −3.0, −0.6, −0.2, 0.1, 0.5, 1.0. Model: A5.

1. Compute $D_6$ against N(0, 1).
2. Compute $D_6$ against $N(\hat\mu, \hat\sigma^2)$.
3. Explain why KS hardly reacts to the −3% day, while AD would.

**Report:** two values of $D_6$, two decisions, one sentence.

In [ ]:
# Your code here


## A7 [Solved]: VaR 1% under two models

**Context.** Normal with μ = 0.05, σ = 1.2; Student-t with ν = 4, location 0.06, scale 0.8.

1. Compute VaR 1% under each model.
2. A bank uses the Normal VaR and observes 20 exceedances in 1000 days; compute the Kupiec statistic.
3. Decide at 5% and interpret.

**Report:** two VaR values, one statistic, one decision.

In [12]:
# Solution
print('Normal VaR 1%:', round(-(0.05 + 1.2 * stats.norm.ppf(0.01)), 3), ' Student-t VaR 1%:', round(-(0.06 + 0.8 * stats.t.ppf(0.01, 4)), 3))
print(kupiec(20, 1000))

Normal VaR 1%: 2.742  Student-t VaR 1%: 2.938
{'x': 20, 'n': 1000, 'rate': 0.02, 'lr': 7.827239152922488, 'p': 0.00514646498249972}


**Interpretation of the result.** Twice the expected number of exceedances: the Normal VaR is too low.

## A8 [Proposed]: too few exceedances, and ES

**Context.** Another bank observes 4 exceedances of its VaR 1% in 1000 days. Model: A7.

1. Compute the Kupiec statistic and decide at 5%.
2. Explain why too few exceedances is also a failure.
3. Compute ES 2.5% of the Normal model of A7.

**Report:** one statistic, one decision, one ES value and one sentence.

In [ ]:
# Your code here


# Part B: real data, precision and interpretation

## B1 [Solved]: which distribution for the BET?

**Question.** Which of the six candidates fits the daily BET returns best, and is the skewness parameter needed?

1. Fit the six candidates by maximum likelihood and report $\ell(\hat\theta)$ of each.
2. Compute AIC, BIC, ΔAIC and the Akaike weights.
3. Test the Student-t against the skewed-t with a likelihood-ratio test.
4. Interpretation: what do the winner and the LR test say about the tails and the skewness of the BET?

**Report:** a table with six rows, one LR test and two sentences.

In [14]:
# Solution
def b1_ranking(name='bet', save=True, chart='ch6_sem_b1'):
    """B1/B2: six candidate models fitted to the daily log returns; the information-criteria table, the LR test of
    the skewness parameter (Student-t inside skewed-t) and of Normal inside GED."""
    r = returns(name)
    x = r.values
    P, tab = fit_table(x)
    out = {'n': len(x), 'first': r.index[0].date().isoformat(), 'last': r.index[-1].date().isoformat(), 'tab': tab,
           'nu': P['Student-t']['nu'], 'lam': P['Skewed-t']['lam'], 'beta': P['GED']['beta'],
           'lr_t_skt': lr_test(P['Student-t']['loglik'], P['Skewed-t']['loglik'], 1),
           'lr_n_ged': lr_test(P['Normal']['loglik'], P['GED']['loglik'], 1),
           'best_aic': min(tab, key=lambda m: tab[m]['aic']), 'best_bic': min(tab, key=lambda m: tab[m]['bic'])}
    if save:
        ms = [m for m in SEM_MODELS if m != 'Normal']
        fig, ax = plt.subplots(figsize=(10, 3.6))
        ax.bar(range(len(ms)), [tab[m]['daic'] for m in ms], color=[MODEL_COL[m] for m in ms], label='_')
        ax.set_xticks(range(len(ms)))
        ax.set_xticklabels(['Normal mixture' if m == 'Mixture' else m for m in ms])
        ax.set_ylabel('Delta AIC')
        ax.set_title(f'{LABELS[name]}: Delta AIC (the Normal law, Delta AIC = {tab["Normal"]["daic"]:,.0f}, is off the scale)')
        ax.axhline(10, color='#CD0000', ls='--', lw=1.0, label='Delta AIC = 10')
        st.legend_outside_bottom(ax, ncol=1, y=-0.14)
        st.check_no_grey(fig)
        st.save_fig(chart)
    return out


b = b1_ranking('bet', save=False)
print(b['best_aic'], b['best_bic'], b['lr_t_skt'])
pd.DataFrame(b['tab']).T.round(3)

NIG NIG {'lr': 0.34770288987783715, 'p': 0.5554163575785768}


,ll,k,aic,bic,daic,dbic,w
Normal,-11701.600,2.0,23407.201,23420.812,2425.192,2411.582,0.000
Student-t,-10495.196,3.0,20996.392,21016.808,14.383,7.578,0.001
Skewed-t,-10495.022,4.0,20998.044,21025.266,16.036,16.036,0.000
GED,-10552.544,3.0,21111.088,21131.504,129.080,122.274,0.000
Mixture,-10563.929,5.0,21137.858,21171.885,155.850,162.655,0.000
NIG,-10487.004,4.0,20982.009,21009.230,0.000,0.000,0.999


**Interpretation of the result.** NIG wins on both criteria; the skewness parameter of the skewed-t is not significant; the Normal law is thousands of AIC points behind.

## B2 [Proposed]: DAX and Bitcoin

**Question.** Do the DAX and Bitcoin choose the same distribution as the BET? Model: B1.

1. Repeat steps 1–3 of B1 for each series.
2. Say whether AIC and BIC agree for each series.
3. Interpretation: why might the best model differ between a stock index and a crypto asset?

**Report:** two tables and three sentences.

In [ ]:
# Your code here


## B3 [Solved]: do the best models pass?

**Question.** Do the Normal and Student-t fits of the S&P 500 pass the KS, Cramér–von Mises and Anderson–Darling tests?

1. Fit both models and compute the three statistics.
2. Compute the naive KS p-value, which treats the parameters as known.
3. Compute parametric bootstrap p-values with B = 99 (simulate, refit, recompute).
4. Draw the QQ plot of the data against both fits.
5. Interpretation: if both models are rejected, what is the test still good for?

**Report:** a table of six statistics with p-values, the QQ plot and two sentences.

In [16]:
# Solution
def b3_gof(name='sp500', B=99, save=True):
    """B3: KS, CvM and AD statistics of the Normal and Student-t fits with the naive KS p-value and parametric
    bootstrap p-values; QQ plots of both fits."""
    rng = np.random.default_rng(SEED)
    x = returns(name).values
    n = len(x)
    out = {'n': n}
    for m in ['Normal', 'Student-t']:
        p = fit_model(m, x)
        obs = edf_stats(cdf(m, x, p))
        sims = {s: [] for s in obs}
        for _ in range(B):
            y = ppf(m, rng.uniform(size=n), p)
            q = fit_model(m, y)
            for s, v in edf_stats(cdf(m, y, q)).items():
                sims[s].append(v)
        out[m] = {'stat': obs, 'p_naive': float(stats.kstwo.sf(obs['ks'], n)),
                  'p_boot': {s: float((1 + np.sum(np.array(sims[s]) >= obs[s])) / (B + 1)) for s in obs},
                  'crit': {s: float(np.quantile(sims[s], 0.95)) for s in obs}, 'params': p}
    if save:
        xs = np.sort(x)
        pe = (np.arange(1, n + 1) - 0.5) / n
        sel = np.unique(np.concatenate([np.linspace(0, n - 1, 400).astype(int), np.arange(30), np.arange(n - 30, n)]))
        fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), sharey=True)
        for ax, m in zip(axes, ['Normal', 'Student-t']):
            q = ppf(m, pe[sel], out[m]['params'])
            ax.plot(q, xs[sel], '.', ms=3.5, color=MODEL_COL[m], label=f'{m} fit')
            ax.plot([xs[0], xs[-1]], [xs[0], xs[-1]], color='black', lw=0.8, ls='--', label='45-degree line')
            ax.set_xlabel('Model quantile (%)')
            ax.set_title(f'{LABELS[name]}: QQ plot, {m}')
            st.legend_outside_bottom(ax, ncol=2, y=-0.18)
        axes[0].set_ylabel('Empirical quantile (%)')
        fig.tight_layout()
        st.check_no_grey(fig)
        st.save_fig('ch6_sem_b3')
    for m in ['Normal', 'Student-t']:
        out[m].pop('params')
    return out


b3_gof('sp500', save=False)

{'n': 6714,
 'Normal': {'stat': {'ks': 0.09336580825968954,
   'cvm': 22.843089304743447,
   'ad': 127.35228474546147},
  'p_naive': 2.1911752136065824e-51,
  'p_boot': {'ks': 0.01, 'cvm': 0.01, 'ad': 0.01},
  'crit': {'ks': 0.010523471491237793,
   'cvm': 0.11943937254850202,
   'ad': 0.7509685262983111}},
 'Student-t': {'stat': {'ks': 0.01837423942556357,
   'cvm': 0.5680789186058598,
   'ad': 4.694287158374209},
  'p_naive': 0.021221159342743864,
  'p_boot': {'ks': 0.01, 'cvm': 0.01, 'ad': 0.01},
  'crit': {'ks': 0.009188003169287683,
   'cvm': 0.08262056248053014,
   'ad': 0.6540395857015937}}}

**Interpretation of the result.** With more than 6000 days every i.i.d. model is rejected; the statistics still rank the models, and the QQ plot shows where each one fails.

## B4 [Proposed]: crash or data error? Banca Transilvania

**Question.** Which of the largest TLV returns are real, and does removing the wrong ones change the chosen model? Model: B1.

1. List the three largest absolute returns with their dates.
2. For each, compare the close and the adjusted close around the date, and the BET return of the same day.
3. Decide which days are data errors and remove only those.
4. Refit five candidates; compare the kurtosis, the Student-t ν̂, the AD statistic and the AIC winner before and after.
5. Interpretation: what would have happened if you had removed all three days?

**Report:** a table of three days with your verdict, a before/after table and two sentences.

In [ ]:
# Your code here


## B5 [Solved]: VaR 1% out of sample for the DAX

**Question.** Does the model chosen by AIC on 2000–2019 also give the best VaR 1% on 2020–2026?

1. Fit the six candidates on the estimation days and rank them by AIC.
2. Compute each model's mean log score on the test days.
3. Compute each model's VaR 1%, count its exceedances on the test days and run the Kupiec test.
4. Add historical simulation and the VaR averaged with the Akaike weights.
5. Interpretation: which model would you choose for VaR 1%, and why?

**Report:** one table, the chart and two sentences.

In [18]:
# Solution
def b5_oos(name='dax', split=SPLIT, models=SEM_MODELS, save=True, chart='ch6_sem_b5'):
    """B5/B6: estimation until `split`, test afterwards: out-of-sample mean log score, VaR 1% exceedances (Kupiec),
    pinball loss, and VaR 1% averaged with the Akaike weights of the estimation sample."""
    r = returns(name)
    est, test = r.loc[:split].values, r.loc[split:].iloc[1:].values
    P, tab = fit_table(est, models)
    out = {'n_est': len(est), 'n_test': len(test), 'first': r.index[0].date().isoformat(), 'last': r.index[-1].date().isoformat(),
           'tab': tab, 'best_aic': min(tab, key=lambda m: tab[m]['aic'])}
    for m in models:
        q = float(np.ravel(ppf(m, ALPHA_VAR, P[m]))[0])
        out[m] = {'logscore': float(np.mean(logpdf(m, test, P[m]))), 'var1': -q, 'kupiec': kupiec(int(np.sum(test < q)), len(test)),
                  'pinball': pinball(test, q)}
    qh = float(np.quantile(est, ALPHA_VAR))
    out['Historical'] = {'var1': -qh, 'kupiec': kupiec(int(np.sum(test < qh)), len(test)), 'pinball': pinball(test, qh)}
    qa = -sum(tab[m]['w'] * out[m]['var1'] for m in models)
    out['Averaged'] = {'var1': -qa, 'kupiec': kupiec(int(np.sum(test < qa)), len(test)), 'pinball': pinball(test, qa)}
    out['best_logscore'] = max(models, key=lambda m: out[m]['logscore'])
    out['best_pinball'] = min(list(models) + ['Historical', 'Averaged'], key=lambda m: out[m]['pinball'])
    out['test_var1'] = float(-np.quantile(test, ALPHA_VAR))
    if save:
        fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.8))
        axes[0].bar(range(len(models)), [out[m]['logscore'] - out['Normal']['logscore'] for m in models],
                    color=[MODEL_COL[m] for m in models])
        axes[0].set_xticks(range(len(models)))
        axes[0].set_xticklabels(['Mixture' if m == 'Mixture' else m for m in models], rotation=20)
        axes[0].set_title('Test log score minus Normal (per day)')
        ms = models + ['Historical']
        axes[1].bar(range(len(ms)), [100 * out[m]['kupiec']['rate'] for m in ms], color=[MODEL_COL[m] for m in ms])
        axes[1].axhline(1, color='black', ls='--', lw=0.9, label='target 1%')
        axes[1].set_xticks(range(len(ms)))
        axes[1].set_xticklabels(ms, rotation=20)
        axes[1].set_title('VaR 1% exceedances on the test days (%)')
        st.legend_outside_bottom(axes[1], ncol=1, y=-0.3)
        fig.tight_layout()
        st.check_no_grey(fig)
        st.save_fig(chart)
    return out


b = b5_oos('dax', save=False)
print(b['best_aic'], b['best_logscore'], b['best_pinball'])
pd.DataFrame({m: {'logscore': b[m].get('logscore'), 'var1': b[m]['var1'], 'x': b[m]['kupiec']['x'], 'p': b[m]['kupiec']['p'], 'pinball': b[m]['pinball']} for m in SEM_MODELS + ['Historical', 'Averaged']}).T.round(4)

NIG NIG GED


,logscore,var1,x,p,pinball
Normal,-1.6679,3.3701,27.0,0.0265,0.0537
Student-t,-1.5332,4.0892,11.0,0.1127,0.0523
Skewed-t,-1.5316,4.3755,8.0,0.0136,0.0536
GED,-1.5394,3.9146,18.0,0.8283,0.0521
Mixture,-1.5422,4.3922,8.0,0.0136,0.0537
NIG,-1.5299,4.3765,8.0,0.0136,0.0536
Historical,NaN,4.3342,8.0,0.0136,0.0533
Averaged,NaN,4.3738,8.0,0.0136,0.0536


**Interpretation of the result.** The AIC winner gives the best density but too few exceedances; for VaR 1% alone the GED and the Student-t were closer to the 1% target.

## B6 [Proposed]: VaR 1% out of sample for Bitcoin

**Question.** Does the B5 conclusion hold for Bitcoin? Model: B5.

1. Repeat steps 1–4 of B5.
2. Compare the empirical VaR 1% of the test days with that of the estimation days.
3. Interpretation: why do all heavy-tailed models fail the Kupiec test here, and in which direction?

**Report:** one table and two sentences.

In [ ]:
# Your code here


# Part C: open questions and AI critique

## C1 [Proposed]: does the best model change over time?

**Question.** Is the AIC winner of the S&P 500 and the BET the same in every two-year window? Models: B1, A1.

1. In each window, find the AIC winner, its Akaike weight and the ΔAIC of the runner-up.
2. Count how often each model wins, for each index.
3. Propose one explanation for the changes and one way to test it.
4. Interpretation: should a risk model be re-selected every two years?

**Report:** a table, a count of winners and a plan for a project.

In [ ]:
# Your code here


## C2 [Proposed]: audit an AI answer

**Context.** An AI assistant chose a distribution for the daily S&P 500 log returns (%) since 2000 and claimed:

- (a) the skewed-t has the lowest AIC, so it is significantly better than the Student-t;
- (b) the KS test of the Student-t fit with `scipy.stats.kstest` gives p = 0.021, so the Student-t fits at the 1% level;
- (c) the LR test of Normal vs Student-t uses the χ²(1) critical value 3.84;
- (d) BIC penalises extra parameters less than AIC, because ln(n) is only 8.81;
- (e) the Anderson–Darling test is more sensitive than KS to the fit in the tails;
- (f) since the skewed-t has the best fit, its VaR 1% is the most accurate one.

1. For each statement, say whether it is correct; if not, give the correct statement and the correct number.

**Report:** six verdicts with one line of justification each.

In [ ]:
# Your code here
